In [ ]:
from pathlib import Path

# Find the notebooks whether the kernel starts here or at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/01_graphs.ipynb"

# 2. Finding cores and generating random intersection graphs

This notebook explains how to find a graph's $k$-core and how to generate a
graph from randomly assigned community memberships. We then specialise the
generator to communities with three membership places. The last sections
add configuration-model graphs for comparison, keeping their edge multiplicities.

The setup cell loads the graph classes from `01_graphs.ipynb`. Running this
notebook defines the functions used by the simulation notebook.

In [ ]:
import copy
from collections import deque
from collections.abc import Callable, Sequence
from itertools import combinations
from numbers import Integral

import numpy as np

## The $k$-core by repeated deletion

The $k$-core is the largest part of a graph in which each vertex has at least
$k$ neighbours within that part. We repeatedly remove vertices with too few
neighbours. Since each removal can lower other degrees, one pass is not enough.

The queue is a list of vertices waiting to be removed. It starts with all
vertices whose degree is below $k$. When removing a vertex would lower a
neighbour's degree from $k$ to $k-1$, that neighbour joins the queue. Each
vertex joins at most once. When the queue is empty, the remaining graph is
the $k$-core.

`construct_k_core(graph, k)` works on an independent copy, leaving the input
graph available for comparison. Surviving vertices keep their UUID labels.
The membership records still describe how the graph was generated, including
vertices removed during pruning.

The deletion step takes $O(|V|+|E|)$ work because each vertex is queued at most
once and each edge is removed at most once. For $k=0$, every vertex survives,
including vertices with no neighbours.

In [ ]:
def construct_k_core(graph: Graph, k: int) -> Graph:
    """Return the k-core as a separate Graph without modifying the input."""
    if not isinstance(k, Integral) or k < 0:
        raise ValueError("k must be a nonnegative integer")

    core = copy.deepcopy(graph)
    q = deque(node for node in core.nodes if node.degree < k)

    while len(q) > 0:
        node = q.popleft()

        # Removing this node lowers each neighbour's degree by one.
        # Add neighbours to the queue just before their degree falls below k.
        for neighbour in node.neighbours:
            if neighbour.degree == k:
                q.append(neighbour)

        core.remove_node(node)

    return core

## Building a graph from community memberships

In a random intersection graph (RIG), individuals are connected when they
share a community. We assign memberships first, then turn them into edges.

`construct_random_intersection_graph(n, k, D)` takes the number of individuals
$n$, the number of places in each community $k$, and a sampling function
`D`. Calling `D(n)` returns one nonnegative integer membership count per
individual. Here $k$ describes community size; it is separate from the degree
threshold used in the core algorithm above.

1. Draw the membership counts $(D_1,\ldots,D_n)$. Their total
   $M=\sum_i D_i$ must be divisible by $k$, so that every community can be
   filled. If it is not, redraw the **whole vector**. For independent Poisson
   draws, this samples their joint law conditioned on $M\equiv0\pmod{k}$.
   Adjusting a single count to make the total fit would change that law.
2. Make a list containing individual $i$ exactly $D_i$ times. Each appearance
   is called a **stub**: a placeholder for one membership. Make a second list
   with $k$ places for each of the $M/k$ communities. Shuffle the individual
   stubs and pair the two lists in order to assign each membership to a place.
3. Connect every pair of distinct individuals sharing a community. One
   individual may occupy several places in the same community, but this does
   not create an edge from a vertex to itself. A pair sharing several
   communities still receives only one edge.

Thus a community has exactly $k$ membership places, but it can contain fewer
than $k$ distinct individuals. Turning community memberships into graph edges
is called **projection**. The matching is uniform over the labelled stubs;
the resulting simple graphs need not be equally likely. The graph keeps the
accepted membership counts and all community places for later analysis.

The optional NumPy generator `rng` controls shuffling. Use it inside `D` too
to repeat both the counts and matching from a seed. The `max_attempts` limit
raises an error if no divisible total is found, avoiding an endless loop
for a sampler that always returns an incompatible vector.

In [ ]:
def construct_random_intersection_graph(
    n: int,
    k: int,
    D: Callable[[int], Sequence[int]],
    rng: np.random.Generator | None = None,
    max_attempts: int = 100_000,
) -> Graph:
    """Build a random intersection graph by matching community memberships.

    D(n) supplies one membership count per individual. Redraw all counts
    until their total is divisible by the number of places per community, k.
    The optional rng controls matching; D controls its own random draws.
    Keep the accepted counts and community places on the returned graph.
    """
    if not isinstance(n, Integral) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    if not isinstance(k, Integral) or k < 1:
        raise ValueError("k must be a positive integer")
    if not isinstance(max_attempts, Integral) or max_attempts < 1:
        raise ValueError("max_attempts must be a positive integer")
    if rng is None:
        rng = np.random.default_rng()

    graph = Graph(n)

    # Redraw every count when the total cannot fill complete communities.
    for _ in range(max_attempts):
        d = list(D(n))
        if len(d) != n or any(
            not isinstance(count, Integral) or count < 0 for count in d
        ):
            raise ValueError("D(n) must return n nonnegative integer counts")
        d = [int(count) for count in d]
        if sum(d) % k == 0:
            break
    else:
        raise ValueError(
            f"membership total not divisible by {k} after {max_attempts} attempts"
        )

    m = sum(d) // k
    communities = [[] for _ in range(m)]

    # Each appearance of a vertex represents one membership to assign.
    nodes = list(graph.nodes)
    nodes_by_multiplicity = []
    for i in range(n):
        for _ in range(d[i]):
            nodes_by_multiplicity.append(nodes[i])

    # Each community appears k times, once for each place to be filled.
    communities_by_multiplicity = []
    for community in communities:
        for _ in range(k):
            communities_by_multiplicity.append(community)

    # Shuffling one list gives a uniform random pairing of the two lists.
    rng.shuffle(nodes_by_multiplicity)
    for node, community in zip(nodes_by_multiplicity, communities_by_multiplicity):
        community.append(node)

    # Connect distinct community members, adding each graph edge only once.
    for community in communities:
        for node1, node2 in combinations(community, 2):
            if (node1 != node2) and (node1 not in node2.neighbours):
                graph.add_edge(node1, node2)

    graph.memberships = {node.id: count for node, count in zip(nodes, d)}
    graph.communities = tuple(
        tuple(node.id for node in community) for community in communities
    )
    return graph

## The triangle model: three places per community

`construct_special_triangle_RIG(n, pois_param)` uses the construction above
with $k=3$ and independent Poisson membership draws
$D_i\sim\mathrm{Poi}(\lambda)$ before conditioning their total. A community
with three distinct individuals creates a triangle. Repeated individuals
are still allowed, so some communities produce fewer than three edges.

The argument `pois_param` is $\lambda$, the mean of the Poisson membership
draws. In a large sparse graph, the mean graph
degree is approximately $2\lambda$, since a triangle membership usually
provides two neighbours. Repeated individuals or shared edges can reduce a
vertex's degree below twice its accepted membership count.

Pass `rng=np.random.default_rng(seed)` to repeat the Poisson draws and the
matching. Vertex labels are generated separately by `uuid.uuid4()`, so the
same seed gives the same graph structure and core size with different UUID
labels. This is what it means to reproduce a graph **up to relabelling**.
The order in which a node set is visited can also differ between runs.

In [ ]:
def construct_special_triangle_RIG(
    n: int, pois_param: float, rng: np.random.Generator | None = None
) -> Graph:
    """Build a graph with Poisson memberships and three places per community.

    Use one random generator for both the membership draws and matching.
    Vertex labels are generated separately and do not depend on this seed.
    """
    if not np.isfinite(pois_param) or pois_param < 0:
        raise ValueError("pois_param must be finite and nonnegative")
    if rng is None:
        rng = np.random.default_rng()

    def D(size):
        return rng.poisson(pois_param, size)

    return construct_random_intersection_graph(n, 3, D, rng=rng)

## A comparison graph: pairing individual edge ends

The **configuration model** starts with a degree for each vertex. Give vertex
$i$ exactly $d_i$ edge ends, often called *half-edges* or *stubs*. Shuffle all
ends and pair consecutive ends. Each pair becomes one undirected edge. Every
matching of the labelled ends is equally likely, although the resulting
multigraphs need not have equal probabilities. The total degree must be even.

For our comparison we draw $d_i=2D_i$, with independent
$D_i\sim\operatorname{Poisson}(\lambda)$. This has the same limiting degree law
as the triangle model: a membership usually gives two distinct neighbours.
It is **not** a Poisson law with mean $2\lambda$: only even degrees occur, and
the variance is $4\lambda$, rather than $2\lambda$. Triangle projection can
reduce degrees in a finite graph, so the degree match is asymptotic.
The total configuration-model degree is automatically even; no conditioning
or adjustment is needed.

Two ends from the same vertex make a **self-loop**. Several pairs can connect
the same two vertices, giving **parallel edges**. The main configuration-model
measurement keeps both: a loop contributes two to degree, and each parallel
edge contributes separately. This is the multigraph convention of
[Janson–Łuczak, Section 2](https://arxiv.org/pdf/math/0508453), which is used by
the theory notebook. A set of distinct neighbours cannot record this degree,
so we use lists of integer vertex labels for this part of the project.

We also build a simple graph from the **same matching** by deleting loops and
keeping one edge per vertex pair. Comparing its core with the multigraph core
shows how much these finite-graph conventions matter. Deleting edges this way
is called *erasing*. It changes the degree sequence and does **not** sample a
uniform simple graph conditional on the sampled degrees. In particular,
erasing is different from rejecting the entire matching until it is simple.

In [ ]:
def configuration_adjacency(
    n: int, edges: np.ndarray, erase: bool = False
) -> list[list[int]]:
    """Turn endpoint pairs into undirected neighbour lists.

    Keep repeated neighbours and list each self-loop twice unless erase=True.
    With erasing, omit loops and keep one copy of each undirected edge.
    """
    if not isinstance(n, Integral) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    pairs = np.asarray(edges)
    if pairs.shape == (0,):
        pairs = np.empty((0, 2), dtype=np.int64)
    if pairs.ndim != 2 or pairs.shape[1] != 2:
        raise ValueError("edges must have shape (number of edges, 2)")
    if not np.issubdtype(pairs.dtype, np.integer):
        raise ValueError("edge endpoints must be integer vertex labels")
    if np.any(pairs < 0) or np.any(pairs >= n):
        raise ValueError("edge endpoints must lie between 0 and n - 1")

    adjacency = [[] for _ in range(n)]
    seen = set()
    for left, right in pairs:
        left, right = int(left), int(right)
        if erase:
            pair = (min(left, right), max(left, right))
            if left == right or pair in seen:
                continue
            seen.add(pair)
        adjacency[left].append(right)
        adjacency[right].append(left)
    return adjacency


def construct_configuration_model(
    degrees: Sequence[int], rng: np.random.Generator | None = None
) -> dict:
    """Match degree stubs uniformly and keep both graph conventions.

    Return the endpoint pairs, sampled degrees, multigraph adjacency, and its
    erased simple adjacency. Diagnostics count loops and extra non-loop edges
    beyond one per vertex pair. Both adjacencies use the very same matching.
    """
    sampled_degrees = np.asarray(degrees)
    if sampled_degrees.shape == (0,):
        sampled_degrees = np.empty(0, dtype=np.int64)
    if sampled_degrees.ndim != 1 or not np.issubdtype(
        sampled_degrees.dtype, np.integer
    ):
        raise ValueError("degrees must be a one-dimensional sequence of integers")
    if np.any(sampled_degrees < 0) or np.any(sampled_degrees > np.iinfo(np.int64).max):
        raise ValueError("degrees must be nonnegative integers that fit in int64")
    sampled_degrees = sampled_degrees.astype(np.int64, copy=True)
    if sum(int(degree) for degree in sampled_degrees) % 2:
        raise ValueError("the total degree must be even to pair all edge ends")
    if rng is None:
        rng = np.random.default_rng()

    n = len(sampled_degrees)
    stubs = np.repeat(np.arange(n, dtype=np.int64), sampled_degrees)
    rng.shuffle(stubs)
    edges = stubs.reshape(-1, 2)
    adjacency = configuration_adjacency(n, edges)
    simple_adjacency = configuration_adjacency(n, edges, erase=True)
    loops = int(np.count_nonzero(edges[:, 0] == edges[:, 1]))
    simple_edges = sum(map(len, simple_adjacency)) // 2

    return {
        "adjacency": adjacency,
        "simple_adjacency": simple_adjacency,
        "sampled_degrees": sampled_degrees,
        "edges": edges,
        "loops": loops,
        "parallel_excess": len(edges) - loops - simple_edges,
    }

## Finding a core while keeping edge multiplicities

The deletion rule is unchanged: remove a vertex when its current degree is
below $k$, and continue until no such vertex remains. An adjacency list records
one entry for each edge end, so its length is the multigraph degree. Removing
a vertex lowers a neighbour's degree once per edge between them.

`core_mask` returns one Boolean value per vertex: `True` means that vertex
survives. It does not modify the neighbour lists. We mark a removed vertex
before visiting its neighbours, so its self-loops cannot lower another
vertex's degree. Parallel edges are visited separately, as required. Each
vertex is queued at most once, and the work is linear in vertices plus edges.

The same function also works for simple graphs. `graph_adjacency` converts our
`Graph` objects to integer neighbour lists, which lets us check the two core
implementations against each other on small examples. The labels in these
lists are local indices, not the graph's UUID labels.

In [ ]:
def core_mask(adjacency: Sequence[Sequence[int]], k: int = 3) -> np.ndarray:
    """Return vertices surviving degree pruning in an undirected adjacency list.

    Use lists from configuration_adjacency or graph_adjacency. Repeated entries
    represent parallel edges, and each self-loop must appear twice.
    """
    if not isinstance(k, Integral) or isinstance(k, bool) or k < 0:
        raise ValueError("k must be a nonnegative integer")
    degree = [len(neighbours) for neighbours in adjacency]
    alive = [True] * len(adjacency)
    queue = deque(vertex for vertex, value in enumerate(degree) if value < k)

    while queue:
        vertex = queue.popleft()
        alive[vertex] = False
        for neighbour in adjacency[vertex]:
            if alive[neighbour]:
                degree[neighbour] -= 1
                if degree[neighbour] == k - 1:
                    queue.append(neighbour)
    return np.asarray(alive, dtype=bool)


def graph_adjacency(graph: Graph) -> list[list[int]]:
    """Relabel a simple Graph with local integers without modifying the graph."""
    nodes = list(graph.nodes)
    index = {node: position for position, node in enumerate(nodes)}
    return [[index[neighbour] for neighbour in node.neighbours] for node in nodes]